# M1 Lab 2 — First Eval (LLM-as-a-Judge), v1 proof of concept

Runs Ascend IQ's **Version A (Concise)** vs **Version B (Narrative)** system prompts on **one**
sample market-intel question, then has a judge model score both against the golden-set
criteria in `eval-harness-proof.md` (G1–G4 required, N1–N5 = auto-fail).

**Scope: v1 is a single hardcoded question — no CSV loop yet.** Once this runs cleanly end to
end, v2 will loop this same logic over all 20 rows of `starter-dataset.csv`.

**Setup (Colab secrets):** click the key icon in the left sidebar and add three secrets,
enabling notebook access for each:
```
ANTHROPIC_API_KEY
GOOGLE_API_KEY
LANGSMITH_API_KEY
```

In [ ]:
!pip install -q anthropic google-genai langsmith

In [ ]:
import os
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

# LangSmith tracing — every traced call below shows up in the LangSmith project below.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "ascend-iq-m1-poc"

from langsmith import traceable
import anthropic
from google import genai

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
genai_client = genai.Client(api_key=GOOGLE_API_KEY)

GENERATOR_MODEL = "claude-sonnet-5"  # different family than the judge, avoids self-preference bias
JUDGE_MODEL = "gemini-3.6-flash"

## Test question

Row 1 from `starter-dataset.csv`, hardcoded for this v1 POC. `RETRIEVED_CONTEXT` stands in for
what a real retrieval step would return — the source snippets Ascend IQ is allowed to cite from.

In [ ]:
QUESTION = "How does Corvant Analytics' Enterprise tier pricing compare to Nexlify's Enterprise tier?"

RETRIEVED_CONTEXT = """\
Source: Corvant Analytics pricing page
\"Enterprise: $18,000/year, includes unlimited seats, dedicated CSM, and SOC 2 report access.\"

Source: Nexlify pricing page
\"Enterprise plan: $24,000/year for up to 50 seats; additional seats billed at $200/seat/year.\"

Source: G2 review, Nexlify
\"Great platform but the enterprise tier gets expensive fast once you add seats.\"
"""

USER_MESSAGE = f"""Retrieved context:
{RETRIEVED_CONTEXT}

Question: {QUESTION}"""

SYSTEM_A = """You are Ascend IQ, a market intelligence assistant for VP-level strategists and product
leaders at Fortune 500 companies. You answer questions about competitors' pricing,
positioning, and reviews using only the retrieved source documents provided to you.

Rules:
- Lead with the direct answer or verdict in the first line.
- Follow with a short bulleted list of the supporting facts, each with an inline citation
  (source name) immediately after the claim it supports.
- No preamble, no restated question, no repeated caveats.
- If the retrieved sources don't contain enough information to answer confidently, say so
  explicitly and state what's missing — never infer or estimate a value that isn't sourced."""

SYSTEM_B = """You are Ascend IQ, a market intelligence assistant for VP-level strategists and product
leaders at Fortune 500 companies. You answer questions about competitors' pricing,
positioning, and reviews using only the retrieved source documents provided to you.

Rules:
- State the direct answer or verdict in the first sentence.
- Explain the reasoning in short prose paragraphs, organized by the dimension
  being compared (e.g., fee structure, volume pricing), covering every entity
  asked about on each dimension.
- Cite the source name in parentheses immediately after each factual claim.
- Close with one takeaway sentence that follows only from facts cited above.
  Do not introduce new claims.
- If any value the question requires is absent from the retrieved sources, say
  so and name what is missing. Never infer or estimate an unsourced value."""

## Generate: Version A vs Version B (Claude Sonnet 5)

In [ ]:
@traceable(name="generate_answer")
def generate(system_prompt: str, user_message: str) -> str:
    response = anthropic_client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    # Claude Sonnet 5 can emit a thinking block before the text block, so
    # find the text block instead of assuming content[0].
    for block in response.content:
        if block.type == "text":
            return block.text.strip()
    raise ValueError("No text block found in response.content")


output_a = generate(SYSTEM_A, USER_MESSAGE)
output_b = generate(SYSTEM_B, USER_MESSAGE)

print("=== Version A (Concise) ===")
print(output_a)
print("\n=== Version B (Narrative) ===")
print(output_b)

## Judge: score both against the golden-set criteria (Gemini 3.6 Flash)

Judge model is a **different family** than the generator (Gemini vs. Claude) to avoid
self-preference bias, per `eval-harness-proof.md`.

In [ ]:
GOLDEN_CRITERIA = """Good (PASS requires G1-G4):
- G1. Grounded: every factual claim (figure, name, date, price, quote, attribution) appears
  in the retrieved context, and every conclusion follows from cited facts.
- G2. Answer-first: the first sentence directly answers the question asked, or states that
  the retrieved data can't support an answer.
- G3. Complete: every part of the question is addressed; a comparison covers both entities
  on the same dimensions.
- G4. Gaps named: where the context lacks data the answer needs, the output names what's
  missing and doesn't fill it in.

Never (any one = FAIL):
- N1. A figure, name, or claim not present in the retrieved context.
- N2. An unsupported inference stated as fact.
- N3. Answering about a different entity, segment, or tier than the one asked about.
- N4. A confident answer when the retrieved context is empty or irrelevant.
- N5. Hedging, refusing, or flagging "missing data" that is actually present in the context.
"""

JUDGE_PROMPT = f"""You are grading two candidate answers to the same market-intelligence question
against a strict rubric. Judge each candidate independently against every G and N item, then
declare a winner.

RUBRIC:
{GOLDEN_CRITERIA}

RETRIEVED CONTEXT:
{RETRIEVED_CONTEXT}

QUESTION:
{QUESTION}

CANDIDATE A:
{output_a}

CANDIDATE B:
{output_b}

For each candidate, state PASS or FAIL for G1-G4 and note any N1-N5 violations by number.
Then output a final line exactly in the form: WINNER: A or WINNER: B or WINNER: TIE"""


@traceable(name="judge_verdict")
def judge(prompt: str) -> str:
    response = genai_client.models.generate_content(
        model=JUDGE_MODEL,
        contents=prompt,
    )
    return response.text.strip()


verdict = judge(JUDGE_PROMPT)
print(verdict)

## Next: v2

Once this v1 run is debugged (outputs look right, judge verdict is sane, traces show up in
LangSmith under the `ascend-iq-m1-poc` project), v2 will loop `generate`/`judge` over every row
of `starter-dataset.csv` and aggregate a win-rate + G1-G4 pass-rate per version.